# ServiceCycle Python SDK -- Quickstart

Interactive walkthrough of the same flows as `basic_usage.py`. Set `SC_API_KEY` in your environment before running, or paste a key into the cell below (don't commit it).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
from servicecycle import ServiceCycleClient, ServiceCycleError, NotFoundError

API_KEY = os.environ.get('SC_API_KEY', 'sc_replace_me')
client = ServiceCycleClient(api_key=API_KEY)

## 1. Verify the key
`identity.me()` is a cheap credential health check -- it returns the key's scopes so you can confirm it can do what you're about to ask it to do.

In [ ]:
identity = client.identity.me()
identity

## 2. List assets due for maintenance soon

In [ ]:
from datetime import date, timedelta

due_before = (date.today() + timedelta(days=30)).isoformat()
result = client.assets.list(due_before=due_before, limit=10)
print(result['pagination'])
for asset in result['data']:
    print(asset['equipmentType'], '-', asset['governingCondition'])

## 3. Auto-paginate through every open deficiency
`list_all()` is a generator -- it fetches pages lazily as you iterate, so you never have to manage the page counter yourself.

In [ ]:
count = 0
for d in client.deficiencies.list_all(status='OPEN'):
    count += 1
print(f'{count} open deficiencies total')

## 4. Arc-flash pre-check before issuing energized work
`work_order_precheck` is the one endpoint you should treat as a hard gate in any automation that issues work orders on energized equipment.

In [ ]:
labels = client.arc_flash.list_labels(severity='danger', limit=1)
if labels['data']:
    asset_id = labels['data'][0]['assetId']
    precheck = client.arc_flash.work_order_precheck(asset_id)
    print(precheck)
else:
    print('No danger-severity labels on this account to demo against.')

## 5. Error handling
Every SDK error is a `ServiceCycleError` subclass carrying `.status_code` and `.message`.

In [ ]:
try:
    client.assets.get('00000000-0000-0000-0000-000000000000')
except NotFoundError as e:
    print('Expected 404:', e.message)
except ServiceCycleError as e:
    print(f'API error {e.status_code}: {e.message}')